In [10]:
!pip install pandas
import pandas as pd


# 1. 파일 이름 정확하게 맞추기
# 주의: 왼쪽 탐색기에 보이는 실제 파일 이름과 100% 똑같아야 합니다.
# 파일 이름 복사해서 아래 따옴표 안에 붙여넣는 것을 추천합니다.
file_path = "강원특별자치도 춘천시_버스노선별 시간대별 승하차 인원_20251209.csv"

# 2. 인코딩 안전장치 (가장 에러가 많이 나는 부분)
try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='cp949')

# 3. 데이터 다듬기 (띄어쓰기 지우기)
# 공공데이터 특성상 '남춘천역 '처럼 뒤에 공백이 숨어있는 경우가 많아서 다 지워줍니다.
df['정류장명'] = df['정류장명'].str.strip()
df['노선번호'] = df['노선번호'].astype(str).str.strip()

# 4. 남기고 싶은 타겟 지정 (한림대 통학 루트)
# 팀 회의 결과에 따라 노선과 정류장 이름을 자유롭게 추가/삭제하세요.
target_routes = ['10', '11', '12', '100', '300'] 
target_stops = ['남춘천역', '남춘천역환승센터', '명동입구', '팔호광장', '한림대', '한림대학교', '강원대후문']

# 5. 데이터 다이어트 (필터링 실행)
filtered_df = df[
    df['노선번호'].isin(target_routes) & 
    df['정류장명'].isin(target_stops)
]

# 6. 결과 확인
print("==================================")
print(f"🔥 필터링 전 전체 데이터: {len(df):,}행")
print(f"✅ 한림대 관련 데이터: {len(filtered_df):,}행 추출 완료!")
print("==================================")

# 남은 데이터 표 형태로 5줄만 미리보기
display(filtered_df.head())

# 7. 깃허브용 / 다음 작업용 새 CSV 저장
# 맥, 윈도우 상관없이 한글이 안 깨지는 utf-8-sig 포맷 사용
filtered_df.to_csv("한림대_통학_승하차데이터.csv", index=False, encoding='utf-8-sig')
print(">> '한림대_통학_승하차데이터.csv' 파일 저장이 완료되었습니다.")


zsh:1: command not found: pip
🔥 필터링 전 전체 데이터: 65,535행
✅ 한림대 관련 데이터: 689행 추출 완료!


,수집일자,노선아이디,노선번호,정류장아이디,정류장명,이용시간대,승차건수,하차건수,데이터 기준일자
56,2025-06-25,42403172,100,4240865,강원대후문,9,0,4,2025-12-09
184,2025-06-25,42403177,300,4240010,남춘천역,9,2,50,2025-12-09
188,2025-06-25,42403177,300,4240156,한림대학교,9,10,6,2025-12-09
203,2025-06-25,42403177,300,4240865,강원대후문,9,0,2,2025-12-09
204,2025-06-25,42403177,300,4240866,강원대후문,9,8,0,2025-12-09


>> '한림대_통학_승하차데이터.csv' 파일 저장이 완료되었습니다.


In [11]:
import requests

# 1. 아까 발급받으신 진짜 Client ID와 Secret으로 변경해 주세요.
client_id = "gthv6ddfee"
client_secret = "fPXCfIgHCdzW7XaH3q421qXGLbtaWEpBPrIXbRex"

query = "강원특별자치도 춘천시 한림대학길 1"

# 2. 회원님이 찾아오신 정확한 URL로 변경!
url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"

params = {
    "query": query
}

# 3. 헤더 역시 공식 문서 예시와 동일하게 세팅
headers = {
    "x-ncp-apigw-api-key-id": client_id,
    "x-ncp-apigw-api-key": client_secret,
    "Accept": "application/json"
}

print(f"새로운 URL로 '{query}' 좌표 검색을 시도합니다...")

response = requests.get(url, params=params, headers=headers)

if response.status_code == 200:
    data = response.json()
    if data['meta']['totalCount'] > 0:
        lng = data['addresses'][0]['x'] # 경도 (Longitude)
        lat = data['addresses'][0]['y'] # 위도 (Latitude)
        print(f"✅ 네이버 지도 API 연결 최종 성공! \n- 위도(Latitude, Y): {lat}\n- 경도(Longitude, X): {lng}")
    else:
        print("검색 결과가 없습니다.")
else:
    print(f"❌ API 호출 실패 (HTTP 상태 코드: {response.status_code})")
    print(response.text)


새로운 URL로 '강원특별자치도 춘천시 한림대학길 1' 좌표 검색을 시도합니다...
✅ 네이버 지도 API 연결 최종 성공! 
- 위도(Latitude, Y): 37.8872151
- 경도(Longitude, X): 127.7382498


In [14]:
import requests
import urllib.parse # 한글 이름을 %EA%B4... 형태로 변환하기 위한 모듈

# 1. ⚠️ 주의: 키 앞뒤로 따옴표("")가 잘 닫혀있는지 꼭 확인하세요!
app_key = "lWNZN6c9qbafOyL5qAudP7335IOhHX8E2pvprQ4C"

# 2. 좌표 설정 (공식 문서 예시처럼 따옴표 없는 숫자형으로 입력)
start_lng = 127.7400 # 출발지(정류장) 경도 X
start_lat = 37.8830  # 출발지(정류장) 위도 Y
end_lng = 127.7420   # 도착지(공학관) 경도 X
end_lat = 37.8860    # 도착지(공학관) 위도 Y

# 3. 보행자 경로안내 API 주소
url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

# 4. 한글 이름 깨짐 방지를 위해 공식 예시처럼 URL 인코딩 적용
start_name_encoded = urllib.parse.quote("한림대정류장")
end_name_encoded = urllib.parse.quote("한림대공학관")

# curl 예시와 완벽하게 동일한 데이터 구조
payload = {
    "startX": start_lng,
    "startY": start_lat,
    "endX": end_lng,
    "endY": end_lat,
    "startName": start_name_encoded,
    "endName": end_name_encoded
}

# curl 예시와 완벽하게 동일한 헤더 구조
headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "appKey": app_key
}

print("정류장에서 공학관까지 도보 시간을 계산합니다...")

# API 호출
response = requests.post(url, json=payload, headers=headers)

# 결과 출력
if response.status_code == 200:
    data = response.json()
    
    # JSON 결과에서 총 소요 시간(초)과 총 거리(미터) 추출
    total_time_sec = data['features'][0]['properties']['totalTime']
    total_distance = data['features'][0]['properties']['totalDistance']
    
    # 초를 분으로 변환 (반올림)
    total_time_min = round(total_time_sec / 60)
    
    print(f"✅ 길찾기 완료!")
    print(f"🚶‍♂️ TMAP 기준 실제 걷는 거리: {total_distance}m")
    print(f"⏱ TMAP 기준 예상 소요 시간: 약 {total_time_min}분")
else:
    print(f"❌ API 호출 실패 (상태 코드: {response.status_code})")
    print(response.text)


정류장에서 공학관까지 도보 시간을 계산합니다...
✅ 길찾기 완료!
🚶‍♂️ TMAP 기준 실제 걷는 거리: 590m
⏱ TMAP 기준 예상 소요 시간: 약 8분


In [15]:
import requests

SERVICE_KEY = "a71451c7c7109143b19101cf7ff0fcb04f7278bf1424454a720d9a9d1db98ecb"
CITY_CODE = "32010"  # 춘천시

# 1. 좌표 기반 근접 정류소 조회 API (반경 500m 자동 검색)
NEARBY_STATION_API = "http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList"
# 2. 버스 도착 정보 조회 API
ARRIVAL_API = "http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList"

# 한림대학교 중심 GPS 좌표 (WGS84)
HALLYM_LAT = "37.88635"  # 위도 (gpsLati)
HALLYM_LON = "127.73783"  # 경도 (gpsLong)


def get_nearby_stations(lat, lon):
    """GPS 좌표 기준 500m 이내 정류소 목록 가져오기"""
    url = f"{NEARBY_STATION_API}?serviceKey={SERVICE_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=50&pageNo=1&_type=json"

    res = requests.get(url, timeout=10)
    data = res.json()

    body = data.get("response", {}).get("body", {})
    if not body:
        return []

    items_container = body.get("items")
    if not items_container or not isinstance(items_container, dict):
        return []

    items = items_container.get("item", [])
    if isinstance(items, dict):
        items = [items]

    return items


def get_arrival_info(node_id, node_name):
    """각 정류소별 실시간 버스 도착 정보 조회"""
    url = f"{ARRIVAL_API}?serviceKey={SERVICE_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=20&pageNo=1&_type=json"

    try:
        res = requests.get(url, timeout=10)
        data = res.json()

        body = data.get("response", {}).get("body", {})
        items = body.get("items")

        print(f"\n📍 [{node_name}] (정류소 ID: {node_id})")

        if not items or not isinstance(items, dict):
            print("   -> 현재 운행/도착 예정인 버스가 없습니다.")
            return

        item_list = items.get("item", [])
        if isinstance(item_list, dict):
            item_list = [item_list]

        for item in item_list:
            route_no = item.get("routeno")
            arr_time = item.get("arrtime", 0)
            prev_cnt = item.get("arrprevstationcnt", 0)
            vehicle_tp = item.get("vehicletp", "일반버스")

            min_left = arr_time // 60
            sec_left = arr_time % 60

            print(
                f"  - {route_no}번 ({vehicle_tp}): 약 {min_left}분 {sec_left}초 후 ({prev_cnt}개 전 정류장)"
            )

    except Exception as e:
        print(f"  -> 도착 정보 조회 오류: {e}")


if __name__ == "__main__":
    print("=== 한림대학교 반경 500m 이내 버스 정류소 검색 및 도착 정보 ===")

    stations = get_nearby_stations(HALLYM_LAT, HALLYM_LON)

    if not stations:
        print("반경 500m 이내에 검색된 정류소가 없습니다.")
    else:
        print(
            f"한림대 반경 500m 내에서 총 {len(stations)}개의 정류소를 찾았습니다.\n"
        )
        print("-" * 60)

        for sttn in stations:
            node_id = sttn.get("nodeid")
            node_name = sttn.get("nodenm")
            get_arrival_info(node_id, node_name)

=== 한림대학교 반경 500m 이내 버스 정류소 검색 및 도착 정보 ===
한림대 반경 500m 내에서 총 5개의 정류소를 찾았습니다.

------------------------------------------------------------

📍 [한림대학교] (정류소 ID: CCB250002127)
  - 2번 (저상버스): 약 3분 0초 후 (4개 전 정류장)
  - 300번 (저상버스): 약 10분 0초 후 (13개 전 정류장)

📍 [한림대학교] (정류소 ID: CCB250001275)
  - 2번 (저상버스): 약 12분 0초 후 (19개 전 정류장)
  - 10-1번 (저상버스): 약 14분 0초 후 (21개 전 정류장)
  - 7-2(칠전동경유)번 (저상버스): 약 27분 0초 후 (30개 전 정류장)

📍 [동보빌리지] (정류소 ID: CCB250001547)
  - 2번 (저상버스): 약 2분 0초 후 (3개 전 정류장)

📍 [동보빌리지] (정류소 ID: CCB250001548)
  - 2번 (저상버스): 약 13분 0초 후 (20개 전 정류장)
  - 10-1번 (저상버스): 약 15분 0초 후 (22개 전 정류장)
  - 7-2(칠전동경유)번 (저상버스): 약 28분 0초 후 (31개 전 정류장)

📍 [동산A] (정류소 ID: CCB250001540)
   -> 현재 운행/도착 예정인 버스가 없습니다.


In [3]:
import requests
import urllib.parse

# ==========================================
# 🔑 1. API 키 설정 (회원님의 실제 발급 키 적용 완료)
# ==========================================
# [네이버 Maps API] (지오코딩용)
NAVER_CLIENT_ID = "gthv6ddfee"
NAVER_CLIENT_SECRET = "fPXCfIgHCdzW7XaH3q421qXGLbtaWEpBPrIXbRex"

# [SKT TMAP API] (보행자 길찾기용)
TMAP_APP_KEY = "lWNZN6c9qbafOyL5qAudP7335IOhHX8E2pvprQ4C"

# [공공데이터포털 API] (실시간 버스 도착 정보용 - 팀원분 코드의 키)
PUBLIC_DATA_KEY = "a71451c7c7109143b19101cf7ff0fcb04f7278bf1424454a720d9a9d1db98ecb" 
CITY_CODE = "32010" # 춘천시

# ==========================================
# 🛠 2. 기능별 API 함수 정의
# ==========================================

def get_building_coords(address):
    """네이버 API: 한림대 건물 주소를 위경도 좌표로 변환"""
    url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"
    headers = {
        "x-ncp-apigw-api-key-id": NAVER_CLIENT_ID,
        "x-ncp-apigw-api-key": NAVER_CLIENT_SECRET,
        "Accept": "application/json"
    }
    res = requests.get(url, params={"query": address}, headers=headers)
    if res.status_code == 200:
        data = res.json()
        if data['meta']['totalCount'] > 0:
            # 네이버 데이터 기준: x = 경도(lon), y = 위도(lat)
            return float(data['addresses'][0]['y']), float(data['addresses'][0]['x'])
    return None, None

def get_nearby_stations(lat, lon):
    """공공데이터 API: 건물 좌표 기준 반경 500m 정류장 검색"""
    url = f"http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList?serviceKey={PUBLIC_DATA_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=5&pageNo=1&_type=json"
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if items:
            item_list = items.get("item", [])
            return [item_list] if isinstance(item_list, dict) else item_list
    return []

def get_arrival_info(node_id):
    """공공데이터 API: 해당 정류장의 실시간 버스 도착 정보"""
    url = f"http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList?serviceKey={PUBLIC_DATA_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=5&pageNo=1&_type=json"
    try:
        res = requests.get(url, timeout=10)
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if not items:
            return []
        item_list = items.get("item", [])
        return [item_list] if isinstance(item_list, dict) else item_list
    except:
        return []

def get_walking_time(start_lat, start_lon, start_name, end_lat, end_lon, end_name):
    """TMAP API: 정류장에서 건물까지 걷는 거리와 시간 계산"""
    url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"
    payload = {
        "startX": start_lon, 
        "startY": start_lat,
        "endX": end_lon, 
        "endY": end_lat,
        "startName": urllib.parse.quote(start_name),
        "endName": urllib.parse.quote(end_name)
    }
    headers = {
        "Accept": "application/json", 
        "Content-Type": "application/json", 
        "appKey": TMAP_APP_KEY
    }
    res = requests.post(url, json=payload, headers=headers)
    if res.status_code == 200:
        data = res.json()
        time_sec = data['features'][0]['properties']['totalTime']
        distance_m = data['features'][0]['properties']['totalDistance']
        return round(time_sec / 60), distance_m
    return -1, -1

# ==========================================
# 🏁 3. 메인 실행 로직 (통합 결과 출력)
# ==========================================
if __name__ == "__main__":
    print("🔄 한림대 통학 통합 분석(버스 + 도보)을 시작합니다...\n")
    
    # 목적지 설정
    target_building_name = "공학관"
    target_address = "강원특별자치도 춘천시 한림대학길 1"
    
    print(f"📍 1단계: '{target_building_name}' 주소를 네이버 API로 좌표 변환 중...")
    b_lat, b_lon = get_building_coords(target_address)
    
    if b_lat and b_lon:
        print(f"✅ 좌표 변환 성공 (위도: {b_lat}, 경도: {b_lon})\n")
        
        print("📍 2단계: 공학관 반경 500m 내 버스 정류장 검색 중...")
        # 가독성을 위해 가장 가까운 3개 정류장만 테스트합니다.
        stations = get_nearby_stations(b_lat, b_lon)[:3]
        
        print(f"✅ 주요 정류장 {len(stations)}개 발견! 데이터 결합을 시작합니다.\n")
        print("=" * 55)
        
        # 각 정류장별 버스 도착 시간 & 도보 시간 결합
        for sttn in stations:
            sttn_name = sttn.get("nodenm")
            sttn_lat = float(sttn.get("gpslati"))
            sttn_lon = float(sttn.get("gpslong"))
            node_id = sttn.get("nodeid")
            
            print(f"🚏 [{sttn_name}] 정류장에서 내린다면?")
            
            # 실시간 버스 연동 (팀원 코드)
            arrivals = get_arrival_info(node_id)
            if not arrivals:
                print("  🚍 실시간 다가오는 버스: 현재 운행 정보 없음")
            else:
                for arr in arrivals:
                    route_no = arr.get("routeno")
                    arr_time = arr.get("arrtime", 0)
                    print(f"  🚍 실시간 다가오는 버스: {route_no}번 (약 {arr_time // 60}분 후 도착)")
            
            # TMAP 도보 시간 연동
            walk_min, walk_dist = get_walking_time(sttn_lat, sttn_lon, sttn_name, b_lat, b_lon, target_building_name)
            if walk_min != -1:
                print(f"  🚶‍♂️ {target_building_name}까지 걷는 시간: 약 {walk_min}분 소요 ({walk_dist}m)")
            else:
                print(f"  🚶‍♂️ 도보 경로를 찾을 수 없습니다.")
            
            print("-" * 55)


🔄 한림대 통학 통합 분석(버스 + 도보)을 시작합니다...

📍 1단계: '공학관' 주소를 네이버 API로 좌표 변환 중...
✅ 좌표 변환 성공 (위도: 37.8872151, 경도: 127.7382498)

📍 2단계: 공학관 반경 500m 내 버스 정류장 검색 중...
✅ 주요 정류장 3개 발견! 데이터 결합을 시작합니다.

🚏 [한림대학교] 정류장에서 내린다면?
  🚍 실시간 다가오는 버스: 12번 (약 6분 후 도착)
  🚍 실시간 다가오는 버스: 300번 (약 9분 후 도착)
  🚶‍♂️ 공학관까지 걷는 시간: 약 6분 소요 (502m)
-------------------------------------------------------
🚏 [동산A] 정류장에서 내린다면?
  🚍 실시간 다가오는 버스: 12번 (약 2분 후 도착)
  🚶‍♂️ 공학관까지 걷는 시간: 약 11분 소요 (832m)
-------------------------------------------------------
🚏 [소방서] 정류장에서 내린다면?
  🚍 실시간 다가오는 버스: 8-1번 (약 17분 후 도착)
  🚍 실시간 다가오는 버스: 15번 (약 4분 후 도착)
  🚍 실시간 다가오는 버스: 2번 (약 8분 후 도착)
  🚍 실시간 다가오는 버스: 200-1번 (약 26분 후 도착)
  🚍 실시간 다가오는 버스: 10-1번 (약 6분 후 도착)
  🚶‍♂️ 공학관까지 걷는 시간: 약 11분 소요 (846m)
-------------------------------------------------------


In [6]:
import requests
import urllib.parse
import pandas as pd  # 💡 CSV 파일을 읽기 위해 pandas 추가

# ==========================================
# 🔑 1. API 키 설정 (깃허브 올릴 땐 꼭 지우세요!)
# ==========================================
NAVER_CLIENT_ID = "gthv6ddfee"
NAVER_CLIENT_SECRET = "fPXCfIgHCdzW7XaH3q421qXGLbtaWEpBPrIXbRex"
TMAP_APP_KEY = "lWNZN6c9qbafOyL5qAudP7335IOhHX8E2pvprQ4C"
PUBLIC_DATA_KEY = "a71451c7c7109143b19101cf7ff0fcb04f7278bf1424454a720d9a9d1db98ecb" 
CITY_CODE = "32010" # 춘천시

# ==========================================
# 🏫 2. 교내 정밀 좌표 DB (CSV 파일 자동 연동)
# ==========================================
HALLYM_BUILDINGS = {}

try:
    # 💡 엑셀에서 만든 CSV 파일을 불러옵니다. (한글 깨짐 방지를 위해 인코딩 처리)
    try:
        df_buildings = pd.read_csv('한림대학교 건물별 좌표.csv', encoding='utf-8')
    except UnicodeDecodeError:
        df_buildings = pd.read_csv('한림대학교 건물별 좌표.csv', encoding='cp949') # 엑셀 기본 저장 방식
    
    # 데이터프레임의 각 행을 읽어서 딕셔너리에 자동으로 집어넣습니다.
    # (주의: CSV 파일의 첫 줄(헤더) 이름이 '건물명', '위도(lat)', '경도(lon)' 이어야 합니다)
    for index, row in df_buildings.iterrows():
        building_name = str(row['건물명']).strip()
        HALLYM_BUILDINGS[building_name] = (float(row['위도(lat)']), float(row['경도(lon)']))
        
    print(f"✅ '한림대학교 건물별 좌표.csv'에서 총 {len(HALLYM_BUILDINGS)}개의 건물 데이터를 성공적으로 로드했습니다!\n")

except FileNotFoundError:
    print("❌ '한림대학교 건물별 좌표.csv' 파일을 찾을 수 없습니다. 파일명과 위치를 확인해주세요.")

# ==========================================
# 🛠 3. 기능별 API 함수 정의
# ==========================================
def get_building_coords(address):
    url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"
    headers = {"x-ncp-apigw-api-key-id": NAVER_CLIENT_ID, "x-ncp-apigw-api-key": NAVER_CLIENT_SECRET, "Accept": "application/json"}
    res = requests.get(url, params={"query": address}, headers=headers)
    if res.status_code == 200 and res.json()['meta']['totalCount'] > 0:
        return float(res.json()['addresses'][0]['y']), float(res.json()['addresses'][0]['x'])
    return None, None

def get_nearby_stations(lat, lon):
    url = f"http://apis.data.go.kr/1613000/BusSttnInfoInqireService/getCrdntPrxmtSttnList?serviceKey={PUBLIC_DATA_KEY}&gpsLati={lat}&gpsLong={lon}&numOfRows=5&pageNo=1&_type=json"
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if items:
            item_list = items.get("item", [])
            return [item_list] if isinstance(item_list, dict) else item_list
    return []

def get_arrival_info(node_id):
    url = f"http://apis.data.go.kr/1613000/ArvlInfoInqireService/getSttnAcctoArvlPrearngeInfoList?serviceKey={PUBLIC_DATA_KEY}&cityCode={CITY_CODE}&nodeId={node_id}&numOfRows=5&pageNo=1&_type=json"
    try:
        res = requests.get(url, timeout=10)
        items = res.json().get("response", {}).get("body", {}).get("items", {})
        if not items:
            return []
        item_list = items.get("item", [])
        return [item_list] if isinstance(item_list, dict) else item_list
    except:
        return []

def get_walking_time(start_lat, start_lon, start_name, end_lat, end_lon, end_name):
    url = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"
    payload = {
        "startX": start_lon, "startY": start_lat, "endX": end_lon, "endY": end_lat,
        "startName": urllib.parse.quote(start_name), "endName": urllib.parse.quote(end_name)
    }
    headers = {"Accept": "application/json", "Content-Type": "application/json", "appKey": TMAP_APP_KEY}
    res = requests.post(url, json=payload, headers=headers)
    if res.status_code == 200:
        data = res.json()
        return round(data['features'][0]['properties']['totalTime'] / 60), data['features'][0]['properties']['totalDistance']
    return -1, -1

# ==========================================
# 🏁 4. 메인 실행 로직 (현재 위치 기반 하교 모드)
# ==========================================
if __name__ == "__main__":
    
    # 💡 [핵심] 현재 내 위치를 여기에 입력합니다. (CSV 파일의 '건물명' 중 하나와 똑같이 입력!)
    my_current_location = "CLC" 
    
    print(f"🔄 현재 위치 '{my_current_location}' 주변의 하교/이동 정보를 탐색합니다...\n")

    # 내 위치의 좌표 확인 (CSV에서 불러온 데이터 우선 확인)
    if my_current_location in HALLYM_BUILDINGS:
        my_lat, my_lon = HALLYM_BUILDINGS[my_current_location]
    else:
        my_lat, my_lon = get_building_coords(my_current_location)

    if my_lat and my_lon:
        print(f"📍 내 위치 주변(반경 500m) 정류장을 찾습니다...")
        stations = get_nearby_stations(my_lat, my_lon)[:3]
        
        print("=" * 55)
        
        for sttn in stations:
            sttn_name = sttn.get("nodenm")
            sttn_lat, sttn_lon = float(sttn.get("gpslati")), float(sttn.get("gpslong"))
            node_id = sttn.get("nodeid")
            
            print(f"🚏 [{sttn_name}] 정류장으로 간다면?")
            
            # TMAP 계산: 출발지='내 위치', 도착지='정류장'
            walk_min, walk_dist = get_walking_time(my_lat, my_lon, my_current_location, sttn_lat, sttn_lon, sttn_name)
            
            if walk_min != -1:
                print(f"  🚶‍♂️ 지금 출발하면: 약 {walk_min}분 걸어갑니다. ({walk_dist}m)")
            else:
                print(f"  🚶‍♂️ 도보 경로를 찾을 수 없습니다.")
            
            # 실시간 버스
            arrivals = get_arrival_info(node_id)
            if not arrivals:
                print("  🚍 도착 예정 버스: 운행 정보 없음")
            else:
                for arr in arrivals:
                    arr_min = arr.get('arrtime', 0) // 60
                    print(f"  🚍 {arr.get('routeno')}번 버스: 약 {arr_min}분 후 도착 예정")
                    
                    if walk_min != -1:
                        if walk_min > arr_min:
                            print("     ⚠️ 버스가 먼저 도착합니다! 뛰어야 할 수도 있어요.")
                        elif walk_min <= arr_min:
                            print("     ✅ 여유 있게 걸어가도 탈 수 있습니다.")
                            
            print("-" * 55)
    else:
        print("❌ 내 위치를 찾을 수 없습니다.")


✅ '한림대학교 건물별 좌표.csv'에서 총 13개의 건물 데이터를 성공적으로 로드했습니다!

🔄 현재 위치 'CLC' 주변의 하교/이동 정보를 탐색합니다...

📍 내 위치 주변(반경 500m) 정류장을 찾습니다...
🚏 [한림대학교] 정류장으로 간다면?
  🚶‍♂️ 지금 출발하면: 약 3분 걸어갑니다. (271m)
  🚍 10번 버스: 약 6분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
  🚍 300번 버스: 약 21분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
-------------------------------------------------------
🚏 [한림대학교] 정류장으로 간다면?
  🚶‍♂️ 지금 출발하면: 약 5분 걸어갑니다. (303m)
  🚍 2번 버스: 약 0분 후 도착 예정
     ⚠️ 버스가 먼저 도착합니다! 뛰어야 할 수도 있어요.
  🚍 12번 버스: 약 19분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
  🚍 10-1번 버스: 약 12분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
-------------------------------------------------------
🚏 [동보빌리지] 정류장으로 간다면?
  🚶‍♂️ 지금 출발하면: 약 6분 걸어갑니다. (437m)
  🚍 10번 버스: 약 6분 후 도착 예정
     ✅ 여유 있게 걸어가도 탈 수 있습니다.
  🚍 906(기점송암리)번 버스: 약 1분 후 도착 예정
     ⚠️ 버스가 먼저 도착합니다! 뛰어야 할 수도 있어요.
-------------------------------------------------------
